# STT — 03 · Test d'overfitting sur 10 clips (PC, CPU)

**But :** vérifier que tout le code (forward, loss CTC, backpropagation, optimizer) **peut apprendre**.
On entraîne sur **10 clips seulement** : le modèle doit les **apprendre par cœur**.

Ici, on **veut** de l'overfitting :
- ✅ **normal** : la loss baisse vers 0, le WER et le CER tombent à 0 %, les phrases prédites deviennent identiques aux phrases attendues ;
- ❌ **anormal** : la loss stagne, devient `nan`/`inf`, ou le modèle ne produit que des blanks après des centaines d'itérations → **bug** à chercher avant Kaggle.

Ce test ne dit **rien** sur la généralisation (pas de dev ici) : ce sera le rôle de l'entraînement sur Kaggle.

## 1. Les 10 clips et les fonctions de mesure

- `greedy_decode` : lit la sortie du modèle (règles du CTC : argmax à chaque instant, fusion des répétitions, suppression des blanks).
- `error_rate` : distance d'édition (substitutions + suppressions + insertions) / longueur de la référence → **WER** sur les mots, **CER** sur les caractères.

*Fonctions validées, rangées dans `stt/metrics.py`.*

In [ ]:
import time
import torch
from torch import nn

from stt.dataset import load_split, CommonVoiceDataset, collate_batch
from stt.model import SpeechTransformer
from stt.text import decode
from stt.metrics import greedy_decode, error_rate   # validées, rangées dans stt/

torch.manual_seed(0)
torch.set_num_threads(4)   # les 4 cœurs du PC

# Les 10 premiers clips d'entraînement, regroupés en UN seul batch (toujours le même)
dataset = CommonVoiceDataset(load_split(["data/fr-FR", "data/fr-CA"], "train"))
features, feature_lengths, targets, target_lengths = collate_batch([dataset[i] for i in range(10)])
references = [decode(targets[b, :target_lengths[b]].tolist()) for b in range(10)]
print("Batch :", features.shape, "| phrases de", target_lengths.min().item(), "à", target_lengths.max().item(), "caractères")

## 2. Boucle d'entraînement

Une **itération** = forward → loss CTC → backpropagation → mise à jour des poids.
Ici une itération = une époque (les 10 clips tiennent dans un seul batch).

Affiché toutes les 25 itérations :

| Colonne | Ce que c'est | Normal dans ce test |
|---|---|---|
| `loss` | Erreur CTC sur les 10 clips | Baisse régulièrement vers 0 |
| `grad` | Norme du gradient (taille de la correction) | Pas de pic énorme ; plafonnée à 1 (gradient clipping) |
| `WER` / `CER` | % de mots / de caractères faux | Descendent vers 0 % |
| exemple | Phrase prédite pour le 1er clip | D'abord vide (que des blanks), puis de plus en plus juste |

In [ ]:
model = SpeechTransformer()                       # taille S, 3,6 M paramètres
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
N_STEPS = 400

history = []
start = time.time()
model.train()                                    # dropout actif (mode entraînement)
for step in range(1, N_STEPS + 1):
    # 1. Forward : log-probabilités (B, T', 50) et longueurs réduites
    log_probs, out_lengths = model(features, feature_lengths)
    # 2. Loss CTC : nn.CTCLoss attend le temps en PREMIER -> (T', B, 50)
    loss = ctc_loss(log_probs.transpose(0, 1), targets, out_lengths, target_lengths)
    # 3. Backpropagation : on efface les anciens gradients, puis on calcule les nouveaux
    optimizer.zero_grad()
    loss.backward()
    # Gradient clipping : plafonne la norme à 1 pour éviter un pas trop brutal
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    # 4. Mise à jour des poids
    optimizer.step()

    if step % 25 == 0 or step == 1:
        hypotheses = greedy_decode(log_probs.detach(), out_lengths)
        wer = error_rate(references, hypotheses, "mot")
        cer = error_rate(references, hypotheses, "caractère")
        history.append((step, loss.item(), wer, cer))
        print(f"it {step:3d} | loss {loss.item():7.3f} | grad {grad_norm.item():6.2f} | "
              f"WER {wer:5.1f} % | CER {cer:5.1f} % | {time.time() - start:5.0f} s | « {hypotheses[0][:45]} »")

## 3. Résultat final (mode évaluation, sans dropout)

In [ ]:
model.eval()
with torch.no_grad():
    log_probs, out_lengths = model(features, feature_lengths)
hypotheses = greedy_decode(log_probs, out_lengths)

print(f"WER final : {error_rate(references, hypotheses, 'mot'):.1f} %")
print(f"CER final : {error_rate(references, hypotheses, 'caractère'):.1f} %")
print()
for ref, hyp in list(zip(references, hypotheses))[:3]:
    print("attendu :", ref)
    print("prédit  :", hyp)
    print()